In [4]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

# ------------------------------------------
# STATE SETTINGS
STATE_NAME = "Katsina"
STATE_ID = 81  # from integrity.ng URL
OUTPUT_FILE = f"{STATE_NAME}_LGA_Wards_PollingUnits.xlsx"
# ------------------------------------------

# Updated URLs using the correct state ID
WARD_URL = f"https://integrity.ng/index.php/wards/browse/{STATE_ID}"
UNIT_URL = f"https://integrity.ng/index.php/units/browse/{STATE_ID}"

def fetch_wards():
    wards = []
    for page in range(0, 100):
        url = f"{WARD_URL}?page={page}"
        print(f"Fetching Wards Page {page}...", end=" ")
        resp = requests.get(url)
        if not resp.ok:
            break
        soup = BeautifulSoup(resp.text, "html.parser")
        table = soup.find("table")
        if not table:
            break
        rows = table.find_all("tr")[1:]
        if not rows:
            break
        for tr in rows:
            cols = [td.get_text(strip=True) for td in tr.find_all("td")]
            if len(cols) >= 4:
                wards.append({
                    "LGA": cols[2],
                    "Ward": cols[3]
                })
        time.sleep(0.2)
        print("✅")
    return wards

def fetch_polling_units():
    pus = []
    for page in range(0, 100):
        url = f"{UNIT_URL}?page={page}"
        print(f"Fetching PUs Page {page}...", end=" ")
        resp = requests.get(url)
        if not resp.ok:
            break
        soup = BeautifulSoup(resp.text, "html.parser")
        table = soup.find("table")
        if not table:
            break
        rows = table.find_all("tr")[1:]
        if not rows:
            break
        for tr in rows:
            cols = [td.get_text(strip=True) for td in tr.find_all("td")]
            if len(cols) >= 5:
                pus.append({
                    "LGA": cols[2],
                    "Ward": cols[3],
                    "Polling Station": cols[4]
                })
        time.sleep(0.2)
        print("✅")
    return pus

def main():
    print(f"\n🔍 Fetching Wards for {STATE_NAME}...")
    all_wards = fetch_wards()
    print(f"✅ Total Wards: {len(all_wards)}")

    print(f"\n🔍 Fetching Polling Units for {STATE_NAME}...")
    all_pus = fetch_polling_units()
    print(f"✅ Total Polling Units: {len(all_pus)}")

    # Create LGAs
    lga_list = sorted(set(w["LGA"] for w in all_wards))
    df_lga = pd.DataFrame({"LGA": lga_list})
    df_wards = pd.DataFrame(all_wards).drop_duplicates()
    df_pus = pd.DataFrame(all_pus).drop_duplicates()

    print(f"\n💾 Saving to Excel...")
    with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
        df_lga.to_excel(writer, sheet_name="LGAs", index=False)
        df_wards.to_excel(writer, sheet_name="Wards", index=False)
        df_pus.to_excel(writer, sheet_name="PollingUnits", index=False)
    print(f"✅ Done! Saved as: {OUTPUT_FILE}\n")

if __name__ == "__main__":
    main()




🔍 Fetching Wards for Katsina...
Fetching Wards Page 0... ✅
Fetching Wards Page 1... ✅
Fetching Wards Page 2... ✅
Fetching Wards Page 3... ✅
Fetching Wards Page 4... ✅
Fetching Wards Page 5... ✅
Fetching Wards Page 6... ✅
Fetching Wards Page 7... ✅
Fetching Wards Page 8... ✅
Fetching Wards Page 9... ✅
Fetching Wards Page 10... ✅
Fetching Wards Page 11... ✅
Fetching Wards Page 12... ✅
Fetching Wards Page 13... ✅
Fetching Wards Page 14... ✅
Fetching Wards Page 15... ✅
Fetching Wards Page 16... ✅
Fetching Wards Page 17... ✅
Fetching Wards Page 18... ✅
Fetching Wards Page 19... ✅
Fetching Wards Page 20... ✅
Fetching Wards Page 21... ✅
Fetching Wards Page 22... ✅
Fetching Wards Page 23... ✅
Fetching Wards Page 24... ✅
Fetching Wards Page 25... ✅
Fetching Wards Page 26... ✅
Fetching Wards Page 27... ✅
Fetching Wards Page 28... ✅
Fetching Wards Page 29... ✅
Fetching Wards Page 30... ✅
Fetching Wards Page 31... ✅
Fetching Wards Page 32... ✅
Fetching Wards Page 33... ✅
Fetching Wards Page 34...

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
from requests.exceptions import RequestException

# ------------------------------------------
# SETTINGS
STATE_NAME = "Katsina"
STATE_ID = 81  # From integrity.ng's Katsina state ID
PAGE_LIMIT = 50
OUTPUT_FILE = f"{STATE_NAME}_LGA_Wards_PollingUnits.xlsx"
# ------------------------------------------

# URLs for state-specific pages
WARD_URL = f"https://integrity.ng/index.php/wards/browse/{STATE_ID}"
UNIT_URL = f"https://integrity.ng/index.php/units/browse/{STATE_ID}"

# Retry wrapper
def safe_request(url):
    for attempt in range(3):
        try:
            response = requests.get(url, timeout=10)
            if response.ok:
                return response
        except RequestException as e:
            print(f"⚠️ Error: {e}. Retrying in 5 seconds...")
            time.sleep(5)
    print("❌ Failed after 3 retries.")
    return None

# Scrape wards
def fetch_wards():
    wards = []
    for page in range(PAGE_LIMIT):
        url = f"{WARD_URL}?page={page}"
        print(f"\n🔄 Fetching Wards Page {page}...", end=" ")
        resp = safe_request(url)
        if not resp:
            print("❌ Request failed.")
            break

        soup = BeautifulSoup(resp.text, "html.parser")
        table = soup.find("table")

        if not table:
            print("❌ No <table> found on page.")
            break

        rows = table.find_all("tr")[1:]
        print(f"↪️ {len(rows)} rows found")
        if not rows:
            break

        for row in rows:
            cols = [td.get_text(strip=True) for td in row.find_all("td")]
            if len(cols) >= 4:
                wards.append({
                    "LGA": cols[2],
                    "Ward": cols[3]
                })
        time.sleep(random.uniform(1, 3))  # Delay between requests
    return wards

# Scrape polling units
def fetch_polling_units():
    pus = []
    for page in range(PAGE_LIMIT):
        url = f"{UNIT_URL}?page={page}"
        print(f"\n🔄 Fetching PUs Page {page}...", end=" ")
        resp = safe_request(url)
        if not resp:
            print("❌ Request failed.")
            break

        soup = BeautifulSoup(resp.text, "html.parser")
        table = soup.find("table")

        if not table:
            print("❌ No <table> found on page.")
            break

        rows = table.find_all("tr")[1:]
        print(f"↪️ {len(rows)} rows found")
        if not rows:
            break

        for row in rows:
            cols = [td.get_text(strip=True) for td in row.find_all("td")]
            if len(cols) >= 5:
                pus.append({
                    "LGA": cols[2],
                    "Ward": cols[3],
                    "Polling Station": cols[4]
                })
        time.sleep(random.uniform(1, 3))
    return pus

# Save to Excel
def save_to_excel(lgas, wards, pus):
    df_lga = pd.DataFrame({"LGA": sorted(set(lgas))})
    df_wards = pd.DataFrame(wards).drop_duplicates()
    df_pus = pd.DataFrame(pus).drop_duplicates()

    print("\n💾 Saving to Excel...")
    with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
        df_lga.to_excel(writer, sheet_name=" Plus LGAs", index=False)
        df_wards.to_excel(writer, sheet_name="Wards", index=False)
        df_pus.to_excel(writer, sheet_name="PollingUnits", index=False)
    print(f"✅ Done! File saved as: {OUTPUT_FILE}")

# Main
def main():
    print(f"\n🚀 Starting scrape for: {STATE_NAME}")

    wards = fetch_wards()
    print(f"\n📌 Total Wards Collected: {len(wards)}")

    pus = fetch_polling_units()
    print(f"\n📌 Total Polling Units Collected: {len(pus)}")

    lga_list = [w['LGA'] for w in wards]
    save_to_excel(lga_list, wards, pus)

if __name__ == "__main__":
    main()



🚀 Starting scrape for: Katsina

🔄 Fetching Wards Page 0... ↪️ 100 rows found

🔄 Fetching Wards Page 1... ↪️ 100 rows found

🔄 Fetching Wards Page 2... ↪️ 100 rows found

🔄 Fetching Wards Page 3... ↪️ 100 rows found

🔄 Fetching Wards Page 4... ↪️ 100 rows found

🔄 Fetching Wards Page 5... ↪️ 100 rows found

🔄 Fetching Wards Page 6... ↪️ 100 rows found

🔄 Fetching Wards Page 7... ↪️ 100 rows found

🔄 Fetching Wards Page 8... ↪️ 100 rows found

🔄 Fetching Wards Page 9... ↪️ 100 rows found

🔄 Fetching Wards Page 10... ↪️ 100 rows found

🔄 Fetching Wards Page 11... ↪️ 100 rows found

🔄 Fetching Wards Page 12... ↪️ 100 rows found

🔄 Fetching Wards Page 13... ↪️ 100 rows found

🔄 Fetching Wards Page 14... ↪️ 100 rows found

🔄 Fetching Wards Page 15... ↪️ 100 rows found

🔄 Fetching Wards Page 16... ↪️ 100 rows found

🔄 Fetching Wards Page 17... ↪️ 100 rows found

🔄 Fetching Wards Page 18... ↪️ 100 rows found

🔄 Fetching Wards Page 19... ↪️ 100 rows found

🔄 Fetching Wards Page 20... ↪️ 100 ro

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time

# ------------------------------------------
STATE_NAME = "Katsina"
STATE_ID = 21  # Katsina ID on integrity.ng
PAGE_LIMIT = 50
OUTPUT_FILE = f"{STATE_NAME}_LGA_Wards_PollingUnits.xlsx"

WARD_URL = f"https://integrity.ng/index.php/wards/browse/{STATE_ID}?page="
UNIT_URL = f"https://integrity.ng/index.php/units/browse/{STATE_ID}?page="
# ------------------------------------------

# Configure Selenium headless browser
def init_driver():
    options = Options()
    options.headless = True
    options.add_argument('--disable-gpu')
    options.add_argument("--window-size=1920,1080")
    driver = webdriver.Chrome(options=options)
    return driver

def scrape_table(driver, url, table_type):
    data = []
    for page in range(PAGE_LIMIT):
        full_url = f"{url}{page}"
        print(f"🔄 Loading {table_type} Page {page}...")
        try:
            driver.get(full_url)
            WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.TAG_NAME, "table"))
            )
            table = driver.find_element(By.TAG_NAME, "table")
            rows = table.find_elements(By.TAG_NAME, "tr")[1:]  # Skip header
            print(f"↪️  {len(rows)} rows")

            for row in rows:
                cols = [td.text.strip() for td in row.find_elements(By.TAG_NAME, "td")]
                if table_type == "wards" and len(cols) >= 4:
                    data.append({"LGA": cols[2], "Ward": cols[3]})
                elif table_type == "units" and len(cols) >= 5:
                    data.append({"LGA": cols[2], "Ward": cols[3], "Polling Station": cols[4]})
        except TimeoutException:
            print(f"⚠️ Timeout on page {page}")
            break
        time.sleep(1.5)  # Avoid rate limit
    return data

def save_to_excel(lgas, wards, pus):
    df_lga = pd.DataFrame({"LGA": sorted(set(lgas))})
    df_wards = pd.DataFrame(wards).drop_duplicates()
    df_pus = pd.DataFrame(pus).drop_duplicates()

    print("\n💾 Saving to Excel...")
    with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
        df_lga.to_excel(writer, sheet_name="LGAs", index=False)
        df_wards.to_excel(writer, sheet_name="Wards", index=False)
        df_pus.to_excel(writer, sheet_name="PollingUnits", index=False)
    print(f"✅ Done! File saved as: {OUTPUT_FILE}")

def main():
    print(f"\n🚀 Starting Selenium scrape for {STATE_NAME}")
    driver = init_driver()

    wards = scrape_table(driver, WARD_URL, "wards")
    print(f"📌 Total Wards Collected: {len(wards)}")

    pus = scrape_table(driver, UNIT_URL, "units")
    print(f"📌 Total Polling Units Collected: {len(pus)}")

    driver.quit()

    lga_list = [w["LGA"] for w in wards]
    save_to_excel(lga_list, wards, pus)

if __name__ == "__main__":
    main()
    



🚀 Starting Selenium scrape for Katsina
🔄 Loading wards Page 0...
↪️  100 rows
🔄 Loading wards Page 1...
↪️  100 rows
🔄 Loading wards Page 2...
↪️  100 rows
🔄 Loading wards Page 3...
↪️  100 rows
🔄 Loading wards Page 4...
↪️  100 rows
🔄 Loading wards Page 5...
↪️  100 rows


In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# URL of polling units page
url = "https://integrity.ng/index.php/units/browse/"

# Fetch page content
response = requests.get(url)
soup = BeautifulSoup(response.content, "html.parser")

# Find the table rows (skip header)
rows = soup.select("table tbody tr")

data = []
for row in rows:
    cols = row.find_all("td")
    if len(cols) >= 7:
        state = cols[0].text.strip()
        lga = cols[1].text.strip()
        ward = cols[2].text.strip()
        polling_station = cols[3].text.strip()
        code = cols[4].text.strip()
        agent_phone = cols[5].text.strip()
        pvc = cols[6].text.strip()
        data.append([state, lga, ward, polling_station, code, agent_phone, pvc])

# Create DataFrame and save to CSV
df = pd.DataFrame(data, columns=["State", "LGA", "Ward", "Polling Station", "Code", "Agent Phone", "PVC"])
df.to_csv("polling_units_integrity.csv", index=False)
print("✅ Saved polling_units_integrity.csv with", len(df), "entries.")


✅ Saved polling_units_integrity.csv with 100 entries.


In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

base_url = "https://integrity.ng/index.php/units/browse?page={}"
all_data = []
page = 1

while True:
    print(f"Scraping page {page}...")
    response = requests.get(base_url.format(page))
    soup = BeautifulSoup(response.content, "html.parser")
    
    # Get table rows
    rows = soup.select("table tbody tr")
    if not rows:
        print("No more rows found. Stopping.")
        break

    for row in rows:
        cols = row.find_all("td")
        if len(cols) >= 7:
            state = cols[0].text.strip()
            lga = cols[1].text.strip()
            ward = cols[2].text.strip()
            polling_station = cols[3].text.strip()
            code = cols[4].text.strip()
            agent_phone = cols[5].text.strip()
            pvc = cols[6].text.strip()
            all_data.append([state, lga, ward, polling_station, code, agent_phone, pvc])

    page += 1

# Save to CSV
df = pd.DataFrame(all_data, columns=["State", "LGA", "Ward", "Polling Station", "Code", "Agent Phone", "PVC"])
df.to_csv("polling_units_integrity_all_pages.csv", index=False)
print(f"✅ Done. Saved {len(df)} records to polling_units_integrity_all_pages.csv")


Scraping page 1...
Scraping page 2...
Scraping page 3...
Scraping page 4...
Scraping page 5...
Scraping page 6...
Scraping page 7...
Scraping page 8...
Scraping page 9...
Scraping page 10...
Scraping page 11...
Scraping page 12...
Scraping page 13...
Scraping page 14...
Scraping page 15...
Scraping page 16...
Scraping page 17...
Scraping page 18...
Scraping page 19...
Scraping page 20...
Scraping page 21...
Scraping page 22...
Scraping page 23...
Scraping page 24...
Scraping page 25...
Scraping page 26...
Scraping page 27...
Scraping page 28...
Scraping page 29...
Scraping page 30...
Scraping page 31...
Scraping page 32...
Scraping page 33...
Scraping page 34...
Scraping page 35...
Scraping page 36...
Scraping page 37...
Scraping page 38...
Scraping page 39...
Scraping page 40...
Scraping page 41...
Scraping page 42...
Scraping page 43...
Scraping page 44...
Scraping page 45...
Scraping page 46...
Scraping page 47...
Scraping page 48...
Scraping page 49...
Scraping page 50...
Scraping 

In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

# Base URL pattern
base_url = "https://integrity.ng/index.php/units/browse/"
page = 1
data = []

while True:
    url = f"{base_url}{page}"
    print(f"🔍 Scraping page {page} -> {url}")
    response = requests.get(url)
    
    if response.status_code != 200:
        print(f"❌ Failed to load page {page}. Status code:", response.status_code)
        break

    soup = BeautifulSoup(response.content, "html.parser")
    rows = soup.select("table tbody tr")

    if not rows:
        print("✅ No more data found. Stopping.")
        break

    for row in rows:
        cols = row.find_all("td")
        if len(cols) >= 7:
            state = cols[0].text.strip()
            lga = cols[1].text.strip()
            ward = cols[2].text.strip()
            polling_station = cols[3].text.strip()
            code = cols[4].text.strip()
            agent_phone = cols[5].text.strip()
            pvc = cols[6].text.strip()
            data.append([state, lga, ward, polling_station, code, agent_phone, pvc])

    print(f"✅ Page {page} scraped successfully. Waiting 2 seconds before next page...")
    page += 1
    time.sleep(2)  # polite delay

# Create DataFrame and save to CSV
df = pd.DataFrame(data, columns=["State", "LGA", "Ward", "Polling Station", "Code", "Agent Phone", "PVC"])
df.to_csv("polling_units_integrity_all_pages.csv", index=False)
print(f"🎉 Done! Saved polling_units_integrity_all_pages.csv with {len(df)} total entries.")


🔍 Scraping page 1 -> https://integrity.ng/index.php/units/browse/1
✅ Page 1 scraped successfully. Waiting 2 seconds before next page...
🔍 Scraping page 2 -> https://integrity.ng/index.php/units/browse/2
✅ Page 2 scraped successfully. Waiting 2 seconds before next page...
🔍 Scraping page 3 -> https://integrity.ng/index.php/units/browse/3
✅ Page 3 scraped successfully. Waiting 2 seconds before next page...
🔍 Scraping page 4 -> https://integrity.ng/index.php/units/browse/4
✅ Page 4 scraped successfully. Waiting 2 seconds before next page...
🔍 Scraping page 5 -> https://integrity.ng/index.php/units/browse/5
✅ Page 5 scraped successfully. Waiting 2 seconds before next page...
🔍 Scraping page 6 -> https://integrity.ng/index.php/units/browse/6
✅ Page 6 scraped successfully. Waiting 2 seconds before next page...
🔍 Scraping page 7 -> https://integrity.ng/index.php/units/browse/7
✅ Page 7 scraped successfully. Waiting 2 seconds before next page...
🔍 Scraping page 8 -> https://integrity.ng/index.

In [5]:
import pandas as pd
import os

# === 1. Load dataset ===
file_path = "polling_units_integrity_all_pages.csv"
data = pd.read_csv(file_path)

# === 2. Define column names ===
state_col = "State"
lga_col = "LGA"
ward_col = "Wards"
pu_col = "Polling Station"  # Update this if different in your data

# === 3. Clean up text columns ===
for col in [state_col, lga_col, ward_col, pu_col]:
    data[col] = data[col].astype(str).str.strip()

# === 4. Output directory ===
output_dir = "OUTPUT_STATES"
os.makedirs(output_dir, exist_ok=True)

# === 5. Helper: clean file name ===
def clean_name(name):
    return "".join(c for c in name if c.isalnum() or c in (' ', '_', '-')).strip().replace(" ", "_")

# === 6. Group by State and write Excel ===
for state_name, state_df in data.groupby(state_col):
    cleaned_state = clean_name(state_name)
    output_path = os.path.join(output_dir, f"{cleaned_state}.xlsx")

    # --- Sheet 1: State, LGA ---
    sheet1_df = state_df[[state_col, lga_col]].drop_duplicates().sort_values(by=[state_col, lga_col])

    # --- Sheet 2: LGA, Ward ---
    sheet2_df = state_df[[lga_col, ward_col]].drop_duplicates().sort_values(by=[lga_col, ward_col])

    # --- Sheet 3: LGA, Ward, Polling Unit ---
    sheet3_df = state_df[[lga_col, ward_col, pu_col]].drop_duplicates().sort_values(by=[lga_col, ward_col, pu_col])

    # --- Write to Excel ---
    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        sheet1_df.to_excel(writer, index=False, sheet_name='State_LGA')
        sheet2_df.to_excel(writer, index=False, sheet_name='LGA_Wards')
        sheet3_df.to_excel(writer, index=False, sheet_name='LGA_Wards_PUs')

    print(f"✅ Saved Excel file for state: {state_name} → {output_path}")

print("\n🎯 Done! One Excel file per state created with 3 sheets each.")


✅ Saved Excel file for state: ABIA → OUTPUT_STATES\ABIA.xlsx
✅ Saved Excel file for state: ADAMAWA → OUTPUT_STATES\ADAMAWA.xlsx
✅ Saved Excel file for state: AKWA IBOM → OUTPUT_STATES\AKWA_IBOM.xlsx
✅ Saved Excel file for state: ANAMBRA → OUTPUT_STATES\ANAMBRA.xlsx
✅ Saved Excel file for state: BAUCHI → OUTPUT_STATES\BAUCHI.xlsx
✅ Saved Excel file for state: BAYELSA → OUTPUT_STATES\BAYELSA.xlsx
✅ Saved Excel file for state: BENUE → OUTPUT_STATES\BENUE.xlsx
✅ Saved Excel file for state: BORNO → OUTPUT_STATES\BORNO.xlsx
✅ Saved Excel file for state: CROSS RIVER → OUTPUT_STATES\CROSS_RIVER.xlsx
✅ Saved Excel file for state: DELTA → OUTPUT_STATES\DELTA.xlsx
✅ Saved Excel file for state: EBONYI → OUTPUT_STATES\EBONYI.xlsx
✅ Saved Excel file for state: EDO → OUTPUT_STATES\EDO.xlsx
✅ Saved Excel file for state: EKITI → OUTPUT_STATES\EKITI.xlsx
✅ Saved Excel file for state: ENUGU → OUTPUT_STATES\ENUGU.xlsx
✅ Saved Excel file for state: FCT → OUTPUT_STATES\FCT.xlsx
✅ Saved Excel file for state:

In [3]:
import os
import pandas as pd

# === CONFIGURATION ===
input_folder = "C:/Users/JARE WORKS/Documents/Jare Js journey/INEC_PollingUnits"   # Folder where your .xlsx files are stored
state_name = "ABIA"             # Change this to the state you want to process
output_file = f"{state_name}_Combined.xlsx"

# === CONTAINERS ===
sheet1_data = []  # STATE, LGA
sheet2_data = []  # STATE, LGA, WARD
sheet3_data = []  # LGA, WARD, Polling Unit Code

# === PROCESS FILES FOR THE SELECTED STATE ===
for file in os.listdir(input_folder):
    if file.endswith(".xlsx") and state_name.lower() in file.lower():
        file_path = os.path.join(input_folder, file)
        try:
            df = pd.read_excel(file_path)
            df.columns = [col.strip().lower() for col in df.columns]

            # Identify columns
            state_col = next((c for c in df.columns if "state" in c), None)
            lga_col = next((c for c in df.columns if "lga" in c), None)
            ward_col = next((c for c in df.columns if "ward" in c), None)
            code_col = next((c for c in df.columns if "code" in c or "unit" in c), None)

            # Extract relevant columns
            if state_col and lga_col:
                sheet1_data.append(df[[state_col, lga_col]].drop_duplicates())

            if state_col and lga_col and ward_col:
                sheet2_data.append(df[[state_col, lga_col, ward_col]].drop_duplicates())

            if lga_col and ward_col and code_col:
                sheet3_data.append(df[[lga_col, ward_col, code_col]].drop_duplicates())

            print(f"✅ Processed: {file}")

        except Exception as e:
            print(f"❌ Error processing {file}: {e}")

# === MERGE AND SAVE ===
if sheet1_data:
    sheet1_df = pd.concat(sheet1_data, ignore_index=True).drop_duplicates()
else:
    sheet1_df = pd.DataFrame(columns=["state", "lga"])

if sheet2_data:
    sheet2_df = pd.concat(sheet2_data, ignore_index=True).drop_duplicates()
else:
    sheet2_df = pd.DataFrame(columns=["state", "lga", "ward"])

if sheet3_data:
    sheet3_df = pd.concat(sheet3_data, ignore_index=True).drop_duplicates()
else:
    sheet3_df = pd.DataFrame(columns=["lga", "ward", "polling_unit_code"])

# === SAVE TO EXCEL ===
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    sheet1_df.to_excel(writer, index=False, sheet_name="State_LGA")
    sheet2_df.to_excel(writer, index=False, sheet_name="State_LGA_Wards")
    sheet3_df.to_excel(writer, index=False, sheet_name="LGA_Wards_Units")

print(f"\n✅ {state_name} data saved to '{output_file}' successfully!")


✅ Processed: ABIA_polling_units.xlsx

✅ ABIA data saved to 'ABIA_Combined.xlsx' successfully!


In [4]:
import os
import pandas as pd

# === Configuration ===
input_folder = "INEC_PollingUnits"           # Original scraped files
output_folder = "INEC_PollingUnits_Cleaned"  # New formatted files
os.makedirs(output_folder, exist_ok=True)

for file in os.listdir(input_folder):
    if file.endswith(".xlsx"):
        file_path = os.path.join(input_folder, file)
        print(f"\nProcessing {file}...")

        # Load Excel sheets
        xls = pd.ExcelFile(file_path)
        df1 = pd.read_excel(xls, "State_LGA")
        df2 = pd.read_excel(xls, "State_LGA_Wards")
        df3 = pd.read_excel(xls, "Polling_Units")

        # --- Ensure columns exist ---
        for col in ["State", "LGA", "Ward"]:
            if col not in df3.columns:
                df3[col] = ""

        # --- Sort by State, LGA, Ward for proper grouping ---
        df3_sorted = df3.sort_values(by=["State", "LGA", "Ward"], ignore_index=True)

        # --- Apply visual formatting (clear repeated State/LGA/Ward) ---
        last_state, last_lga, last_ward = None, None, None

        for i in range(len(df3_sorted)):
            state = df3_sorted.at[i, "State"]
            lga = df3_sorted.at[i, "LGA"]
            ward = df3_sorted.at[i, "Ward"]

            # Clear repeated State
            if state == last_state:
                df3_sorted.at[i, "State"] = ""
            else:
                last_state = state

            # Clear repeated LGA (only when State hasn't changed)
            if lga == last_lga and state == "":
                df3_sorted.at[i, "LGA"] = ""
            else:
                last_lga = lga

            # Clear repeated Ward (only when State & LGA haven't changed)
            if ward == last_ward and lga == "":
                df3_sorted.at[i, "Ward"] = ""
            else:
                last_ward = ward

        # --- Save the cleaned result to new Excel file ---
        cleaned_path = os.path.join(output_folder, file)
        with pd.ExcelWriter(cleaned_path, engine="openpyxl") as writer:
            df1.to_excel(writer, index=False, sheet_name="State_LGA")
            df2.to_excel(writer, index=False, sheet_name="State_LGA_Wards")
            df3_sorted.to_excel(writer, index=False, sheet_name="Polling_Units_Cleaned")

        print(f"✅ Cleaned visual formatting saved → {cleaned_path}")

print("\n🎯 All files formatted successfully (State, LGA, and Ward).")



Processing ABIA_polling_units.xlsx...
✅ Cleaned visual formatting saved → INEC_PollingUnits_Cleaned\ABIA_polling_units.xlsx

Processing ADAMAWA_polling_units.xlsx...
✅ Cleaned visual formatting saved → INEC_PollingUnits_Cleaned\ADAMAWA_polling_units.xlsx

Processing AKWA IBOM_polling_units.xlsx...
✅ Cleaned visual formatting saved → INEC_PollingUnits_Cleaned\AKWA IBOM_polling_units.xlsx

Processing ANAMBRA_polling_units.xlsx...
✅ Cleaned visual formatting saved → INEC_PollingUnits_Cleaned\ANAMBRA_polling_units.xlsx

Processing BORNO_polling_units.xlsx...
✅ Cleaned visual formatting saved → INEC_PollingUnits_Cleaned\BORNO_polling_units.xlsx

Processing CROSS RIVER_polling_units.xlsx...
✅ Cleaned visual formatting saved → INEC_PollingUnits_Cleaned\CROSS RIVER_polling_units.xlsx

Processing DELTA_polling_units.xlsx...
✅ Cleaned visual formatting saved → INEC_PollingUnits_Cleaned\DELTA_polling_units.xlsx

Processing EBONYI_polling_units.xlsx...
✅ Cleaned visual formatting saved → INEC_Pol